# Batch Inference Runner for 2D Cell Division Detection

This notebook is the hands-on companion to our project. Run it when you want a guided, reproducible way to execute the full DARE2d pipeline (inference -> consensus -> plots) on your own `.tif` stacks.

**Workflow**
1. Update the path variables in Cell 2 (or export `DARE2D_BASE_DIR`) and activate the `dare2d` environment.
2. Run the remaining cells in order: data discovery, multi-model inference, postprocessing, and plotting.
3. Inspect the generated folders (`output/`, `postprocessed_results/`, `analysis_plots/`).

> Need installation details, model descriptions, or troubleshooting tips? See `README.md` so this notebook can stay lean.

In [ ]:
# Setup: Import Libraries and Configure Paths

# Import required Python libraries
import os
import sys
import subprocess
import glob
from pathlib import Path

# Configuration Section
# base_dir = root directory of the DARE2d project. Resolve it robustly so this notebook
# works whether it is launched from the repo root or from notebooks/: honour
# DARE2D_BASE_DIR if set, otherwise walk up from the cwd to the folder holding setup.py.
def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for cand in (start, *start.parents):
        if (cand / "setup.py").exists():
            return cand
    return start

base_dir = Path(os.environ.get("DARE2D_BASE_DIR", str(_find_repo_root(Path.cwd()))))
os.chdir(base_dir)  # inference/postprocessing run repo-root-relative commands below

# Define subdirectories relative to base_dir
input_dir = base_dir / "input"  # Directory containing input .tif images
output_dir = base_dir / "output"  # Directory for saving inference results

# Checkpoint directories. Two layouts exist: the plugin's "DARE2D download data" button (and the
# README "Models & data" section) fills the DEMO layout
#     models/demo/neuroepithelium/{regression,segmentation}_checkpoints/
# whereas the legacy CLI layout is <repo-root>/{regression,segmentation}_checkpoints/. Prefer whichever
# is present so a fresh demo install "just works"; an explicit env override always wins.
def _resolve_ckpt_dir(env_var: str, demo_rel: Path, legacy_name: str) -> Path:
    if os.environ.get(env_var):
        return Path(os.environ[env_var])
    demo = base_dir / demo_rel
    legacy = base_dir / legacy_name
    return demo if demo.exists() else legacy  # fall back to legacy; its name drives a clear error later

reg_dir = _resolve_ckpt_dir(
    "DARE2D_REG_DIR", Path("models") / "demo" / "neuroepithelium" / "regression_checkpoints",
    "regression_checkpoints")  # Regression model checkpoints
seg_dir = _resolve_ckpt_dir(
    "DARE2D_SEG_DIR", Path("models") / "demo" / "neuroepithelium" / "segmentation_checkpoints",
    "segmentation_checkpoints")  # Segmentation model checkpoints

# Verify that the base directory exists
if not base_dir.exists():
    raise FileNotFoundError(f"Base directory not found: {base_dir}. Set DARE2D_BASE_DIR to the repo root.")

print(f"Project root: {base_dir}")
print(f"Input directory: {input_dir}")
print(f"Output directory: {output_dir}")
print(f"Regression checkpoints: {reg_dir}")
print(f"Segmentation checkpoints: {seg_dir}")
print("Setup completed successfully.")

## Data Discovery

In this section, we identify all input images that will be processed. The script expects images in .tif format placed in the input directory.

The following code:
- Scans the input directory for .tif files
- Displays the number of images found
- Lists the image names for verification

In [ ]:
# Data Discovery: Find Input Images

# Discover all .tif images in the input directory
input_images = glob.glob(os.path.join(input_dir, "*.tif"))

if not input_images:
    print("Error: No .tif images found in the input directory!")
    print(f"Input directory: {input_dir}")
    print("Please ensure input images are placed in the correct directory.")
    # In a notebook, you might want to raise an error or stop execution
    raise FileNotFoundError("No input images found")

print(f"Found {len(input_images)} input images:")
for img_path in input_images:
    image_name = Path(img_path).stem
    print(f"  - {image_name}")

# Store image names for later use
image_names = [Path(img).stem for img in input_images]

## Batch Inference Execution

Every image discovered in the previous cell is paired with each of the eight model sets. The loop simply verifies the checkpoints, creates an `output_dir/{image_name}_{n}` folder, and calls `scripts.inference.multistage_detection2d`.

> For background on the model architecture, training splits, or checkpoint organization, keep the README handy—this notebook just wires everything together.

In [ ]:
# Batch Inference: Process All Images with Multiple Model Sets

# Number of model sets to use (1-8)
num_model_sets = 8

# Count image x model-set inferences that actually ran, so a checkpoint-path mistake fails LOUDLY
# below instead of silently reporting success with zero detections.
total_success = 0

# Process each image
for img_path, image_name in zip(input_images, image_names):
    print(f"\n{'='*60}")
    print(f"Processing image: {image_name}")
    print(f"{'='*60}")

    # Run inference with each model set
    for n in range(1, num_model_sets + 1):
        print(f"\n--- Model Set {n} ---")

        # Construct paths to model checkpoints
        # Assumes organization: checkpoints_set_{n}_all_but_target/best.h5
        reg_checkpoint = os.path.join(reg_dir, f"checkpoints_set_{n}_all_but_target", "best.h5")
        seg_checkpoint = os.path.join(seg_dir, f"checkpoints_set_{n}_all_but_target", "best.h5")

        # Verify model files exist
        if not os.path.exists(reg_checkpoint):
            print(f"Warning: Regression checkpoint not found: {reg_checkpoint}")
            continue
        if not os.path.exists(seg_checkpoint):
            print(f"Warning: Segmentation checkpoint not found: {seg_checkpoint}")
            continue

        # Create output directory for this image-model combination
        output_folder = os.path.join(output_dir, f"{image_name}_{n}")
        os.makedirs(output_folder, exist_ok=True)

        # Construct the inference command
        cmd = [
            sys.executable,
            "-m", "scripts.inference.multistage_detection2d",
            "--regression", reg_checkpoint,
            "--segmentation", seg_checkpoint,
            "--img", img_path,
            "--output", output_folder
        ]

        print(f"Running command: {' '.join(cmd)}")

        try:
            # Execute the inference. encoding="utf-8", errors="replace" so progress glyphs / any
            # non-ASCII in the child's output never crash the reader with a locale (cp1252, Windows)
            # UnicodeDecodeError.
            result = subprocess.run(cmd, check=True, capture_output=True,
                                    text=True, encoding="utf-8", errors="replace")
            total_success += 1
            print(f"✓ Successfully completed inference for {image_name} with model set {n}")
            # Optionally print stdout if needed: print(result.stdout)

        except subprocess.CalledProcessError as e:
            print(f"✗ Error running inference for {image_name} with model set {n}")
            print(f"Error details: {e}")
            print(f"Stderr: {e.stderr}")
            continue

# Fail loudly if nothing ran — almost always a checkpoint-path problem (see the "not found" warnings
# above), which otherwise silently produces empty results.
if total_success == 0:
    raise FileNotFoundError(
        "No inference ran: no checkpoints were found for any model set.\n"
        f"  Regression dir: {reg_dir}\n"
        f"  Segmentation dir: {seg_dir}\n"
        "Fetch the demo models with the napari plugin's 'DARE2D download data' button (fills "
        "models/demo/neuroepithelium/...), or set DARE2D_REG_DIR / DARE2D_SEG_DIR to your checkpoint "
        "folders. See the README 'Models & data' section."
    )

print(f"\n{'='*60}")
print("Batch inference completed!")
print(f"Results saved in: {output_dir}")
print("Each image has subdirectories for each model set (1-8)")
print(f"Successful inferences: {total_success} / {len(input_images) * num_model_sets} "
      f"({len(input_images)} images x {num_model_sets} model sets)")
print(f"{'='*60}")

## Postprocessing and Consensus Generation

Once inference completes, we hand the per-model outputs to `scripts/postprocessing/main.py` to cluster detections, perform temporal deduplication, and write timestamped consensus folders inside `postprocess_dir`.

> Detailed parameter explanations (e.g., `eps`, `min_models`, `angle_mode`) and standalone CLI usage examples are documented in the README; the notebook focuses on running the batch pipeline end-to-end.

In [ ]:
# Postprocessing: Aggregate Multi-Model Results

# Create directory for postprocessing outputs
postprocess_dir = os.path.join(base_dir, "postprocessed_results")
os.makedirs(postprocess_dir, exist_ok=True)

# Track resolved output directories (handles timestamped folders created by script)
postprocess_output_dirs = {}

# Postprocessing parameters
# ============================================================================
# eps: Spatial clustering distance threshold (pixels)
# ============================================================================
# Divisions detected within eps pixels of each other are grouped as potential
# duplicates from different models. Adjust based on:
# - Smaller cells: decrease to 5-8 for higher precision
# - Larger cells: increase to 12-15 for better clustering
eps = 10

# min_models: Minimum number of models required for consensus
# Range: 1 (any model) to 8 (unanimous agreement)
min_models = 1

# Total number of models used in the ensemble
num_models = 8

# Angle selection mode
angle_mode = "auto"

print(f"Postprocessing outputs will be saved to: {postprocess_dir}")
print(f"Clustering configuration:")
print(f"  - eps: {eps} pixels")
print(f"  - min_models: {min_models}/{num_models}")
print(f"  - angle_mode: {angle_mode}")

# Process each image with postprocessing
for img_path, image_name in zip(input_images, image_names):
    print(f"\n{'='*60}")
    print(f"Postprocessing image: {image_name}")
    print(f"{'='*60}")

    # Define base directory for this image's postprocessed results (timestamp appended by script)
    image_postprocess_dir = os.path.join(postprocess_dir, image_name)

    # Snapshot existing directories to detect newly created timestamped folders
    base_pattern = f"{image_name}*"
    existing_dirs = {d.name for d in Path(postprocess_dir).glob(base_pattern) if d.is_dir()}

    # Construct postprocessing command
    cmd = [
        sys.executable,
        "scripts/postprocessing/main.py",
        "--output_root", str(output_dir),
        "--image_name", image_name,
        "--image_stack", str(img_path),
        "--save_dir", image_postprocess_dir,
        "--eps", str(eps),
        "--min_models", str(min_models),
        "--num_models", str(num_models),
        "--angle_mode", angle_mode
    ]

    print(f"Running postprocessing command: {' '.join(cmd)}")

    try:
        # Execute postprocessing. encoding="utf-8", errors="replace" so non-ASCII in the child's
        # output never crashes the reader with a locale (cp1252, Windows) UnicodeDecodeError.
        result = subprocess.run(cmd, check=True, capture_output=True,
                                text=True, encoding="utf-8", errors="replace")
        print(f"✓ Successfully completed postprocessing for {image_name}")
        
        # Resolve actual output folder (script appends timestamp to avoid overwriting)
        candidates = [d for d in Path(postprocess_dir).glob(base_pattern) if d.is_dir()]
        new_dirs = [d for d in candidates if d.name not in existing_dirs]
        if new_dirs:
            final_dir = max(new_dirs, key=lambda p: p.stat().st_mtime)
        else:
            final_dir = max(candidates, key=lambda p: p.stat().st_mtime, default=Path(image_postprocess_dir))
        postprocess_output_dirs[image_name] = str(final_dir)
        print(f"Postprocessing outputs stored in: {final_dir}")

    except subprocess.CalledProcessError as e:
        print(f"✗ Error in postprocessing for {image_name}")
        print(f"Error details: {e}")
        print(f"Stderr: {e.stderr}")
        continue

print(f"\n{'='*60}")
print("Postprocessing completed for all images!")
print(f"Results saved in: {postprocess_dir}")
print("Each image has its own subfolder with consensus results.")